# Day 2（9/24）pandas + 股票数据

目标：拉取真实 A 股数据，看懂 K 线，算收益率和均线，画图。逐格运行，完成每格末尾的【练习】。

In [ ]:
# %% 0. 导入库 + 中文显示设置 ------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import akshare as ak

plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang SC", "Heiti SC"]
plt.rcParams["axes.unicode_minus"] = False
print("pandas 版本:", pd.__version__)

In [ ]:
# %% 1. pandas 两个核心概念：Series（一列）和 DataFrame（一张表）-----------------------
# Series：带"标签"的一维数组，可以理解成 Excel 里的一列
s = pd.Series([100.0, 102.5, 101.0], index=["周一", "周二", "周三"])
print(s)
print("周二的价格:", s["周二"])

# DataFrame：多列组成的二维表格，可以理解成一整张 Excel 表
demo = pd.DataFrame({
    "code": ["600519", "000858", "601318"],
    "name": ["贵州茅台", "五粮液", "中国平安"],
    "price": [1450.5, 128.3, 52.1],
})
print(demo)
print(demo["name"])              # 取一列 → Series
print(demo[demo["price"] > 100]) # 按条件筛选行（非常常用！）

# 【练习】给 demo 加一列 "sector"（所属行业），再筛选出价格小于 200 的股票

In [ ]:
# %% 2. 拉取真实股票数据 ------------------------------------------------------------
# stock_zh_a_hist：东方财富的 A 股日线数据
#   symbol  股票代码    period  daily=日线
#   adjust  "qfq"=前复权（消除分红、送股造成的价格跳空，做分析一般都用复权价）
# 网络偶尔不稳定（尤其开着代理时），所以这里写了"重试 + 备用数据源"。
import time

def load_stock(symbol, start="20210101", end="20251231"):
    """返回列为 open/high/low/close/volume/pct_chg、以日期为索引的表格。"""
    for attempt in range(3):                       # 东方财富接口，最多试 3 次
        try:
            df = ak.stock_zh_a_hist(symbol=symbol, period="daily",
                                    start_date=start, end_date=end, adjust="qfq")
            df = df.rename(columns={"日期": "date", "开盘": "open", "收盘": "close", "最高": "high",
                                    "最低": "low", "成交量": "volume", "涨跌幅": "pct_chg"})
            break
        except Exception as e:
            print(f"东方财富第 {attempt + 1} 次失败：{type(e).__name__}")
            time.sleep(2)
    else:                                          # 3 次都失败 → 换新浪接口
        print("改用新浪接口")
        prefix = "sh" if symbol.startswith("6") else "sz"
        df = ak.stock_zh_a_daily(symbol=prefix + symbol, start_date=start, end_date=end, adjust="qfq")
        df["pct_chg"] = df["close"].pct_change() * 100
    df["date"] = pd.to_datetime(df["date"])        # 字符串 → 日期类型
    df = df.set_index("date")                      # 用日期当"行号"，之后可以按日期切片
    return df[["open", "high", "low", "close", "volume", "pct_chg"]]


df = load_stock("600519")
print(df.shape)       # (行数, 列数)
df.head()             # 在 Notebook 里，最后一行的表格会被漂亮地显示出来

# 【练习】把 "600519" 换成你感兴趣的另一只股票（比如 000858 五粮液、601318 中国平安），重新运行
# 【读代码】load_stock 里的 for ... else 是 Python 特有的写法：循环没有被 break 打断时才执行 else

In [ ]:
# %% 3. 看看数据长什么样 ------------------------------------------------------------
print(df.info())           # 每列的类型、有没有缺失值
print(df.describe())       # 每列的统计摘要：均值、标准差、最小、最大……
print(df.loc["2025-03"])   # 按日期切片：取 2025 年 3 月的所有数据

# 【练习】取出 2024 年全年的数据，看看有多少个交易日（提示：len()）

In [ ]:
# %% 4. 看懂 K 线（蜡烛图）----------------------------------------------------------
# 每根 K 线代表一天，由 4 个价格决定：
#   开盘价 open、收盘价 close、最高价 high、最低价 low
#   收盘 > 开盘 → 阳线（A 股习惯用红色），当天上涨
#   收盘 < 开盘 → 阴线（绿色），当天下跌
#   中间粗的"实体"是开盘到收盘的区间，上下细线（影线）是最高 / 最低价
def plot_candles(data, title):
    fig, ax = plt.subplots(figsize=(12, 4))
    for i, (_, row) in enumerate(data.iterrows()):
        color = "red" if row["close"] >= row["open"] else "green"
        ax.plot([i, i], [row["low"], row["high"]], color=color, linewidth=1)          # 影线
        ax.bar(i, row["close"] - row["open"], bottom=row["open"], color=color, width=0.6)  # 实体
    step = max(1, len(data) // 8)
    ax.set_xticks(range(0, len(data), step))
    ax.set_xticklabels(data.index[::step].strftime("%m-%d"))
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


plot_candles(df.tail(60), "最近 60 个交易日 K 线")

# 【练习】数一数最近 60 天里有几根阳线（提示：(df.tail(60)["close"] > df.tail(60)["open"]).sum()）
#        论文①里的"三日 K 线形态"，就是用连续 3 根 K 线的组合来判断趋势，
#        比如"红三兵"= 连续三根阳线且收盘价一天比一天高。试着想想怎么用代码找出它。

In [ ]:
# %% 5. 收益率与均线 ----------------------------------------------------------------
df["ret"] = df["close"].pct_change()               # 日收益率 = 今天/昨天 - 1（向量化，不用写循环）
df["ma5"] = df["close"].rolling(5).mean()          # 5 日均线：最近 5 天收盘价的平均
df["ma20"] = df["close"].rolling(20).mean()        # 20 日均线

print(df[["close", "ret", "ma5", "ma20"]].tail(10))
print("平均日收益率:", round(df["ret"].mean(), 5))
print("日收益率标准差（波动率）:", round(df["ret"].std(), 5))
print("涨得最多的一天:", df["ret"].idxmax().date(), f"{df['ret'].max():.2%}")
print("跌得最多的一天:", df["ret"].idxmin().date(), f"{df['ret'].min():.2%}")

# 【练习】自己算一个 60 日均线 ma60
# 【思考】同一只茅台、同一天（2021-09-27），东方财富前复权数据算出来涨 10.94%，新浪算出来涨 9.50%，
#        而 A 股主板单日涨幅上限是 10%（涨停）。为什么两个数据源不一样？哪个更"真实"？
#        提示：看看 df.loc["2021-09-24":"2021-09-28", ["close", "ret", "pct_chg"]]，再查一查"前复权"是怎么调整历史价格的。
#        ——做量化时，数据本身的坑往往比模型更重要。

In [ ]:
# %% 6. 画图：收盘价 + 均线 + 成交量 --------------------------------------------------
recent = df.loc["2025"]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                               gridspec_kw={"height_ratios": [3, 1]})
ax1.plot(recent.index, recent["close"], label="收盘价", linewidth=1.2)
ax1.plot(recent.index, recent["ma5"], label="MA5", linewidth=0.9)
ax1.plot(recent.index, recent["ma20"], label="MA20", linewidth=0.9)
ax1.legend()
ax1.set_title("2025 年收盘价与均线")
ax2.bar(recent.index, recent["volume"], color="gray")
ax2.set_title("成交量")
plt.tight_layout()
plt.show()

# 【练习】技术分析里，MA5 从下往上穿过 MA20 叫"金叉"（常被视为买入信号）。
#        用下面的代码找出 2025 年的金叉日期，再对照图看看之后股价涨了吗？
golden = (recent["ma5"] > recent["ma20"]) & (recent["ma5"].shift(1) <= recent["ma20"].shift(1))
print("2025 年金叉日期:", recent.index[golden].date.tolist())

In [ ]:
# %% 7. 小挑战：这只股票如果"买入并持有"，收益如何？----------------------------------
start_price = df["close"].iloc[0]
end_price = df["close"].iloc[-1]
print(f"{df.index[0].date()} 买入价 {start_price:.2f}，{df.index[-1].date()} 价格 {end_price:.2f}")
print(f"持有收益率: {end_price / start_price - 1:.2%}")

# 累计收益曲线：(1+r1)(1+r2)...(1+rt) - 1
df["cum_ret"] = (1 + df["ret"].fillna(0)).cumprod() - 1
df["cum_ret"].plot(figsize=(10, 3), title="买入并持有的累计收益率")
plt.tight_layout()
plt.show()

# 【练习】最大回撤 = 从历史最高点跌下来的最大幅度，是衡量风险的重要指标（论文③里的 MDD）。
#        提示：净值 nav = 1 + cum_ret；回撤 = nav / nav.cummax() - 1；最大回撤 = 回撤的最小值